## Test X.ai API (GROQ)

In [1]:
import os
from openai import OpenAI
import openai

openai.api_key = os.getenv("XAI_API_KEY")

client = OpenAI(
  api_key=openai.api_key ,
  base_url="https://api.x.ai/v1",
)

completion = client.chat.completions.create(
  model="grok-3-latest",
  messages=[
    {"role": "system", "content": "You are a PhD-level mathematician."},
    {"role": "user", "content": "What is 2 + 2?"},
  ],
)

print(completion.choices[0].message)

ChatCompletionMessage(content='2 + 2 equals 4.\n\nThis is a basic arithmetic operation where two numbers, each of value 2, are combined to produce a sum of 4. In the context of natural numbers or integers, this result is fundamental and can be understood through counting (e.g., combining two groups of two items results in four items) or through the properties of addition in the number system.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [9]:
from git import Repo
import tempfile
import os

# URL of the repo you want to ingest
REPO_URL = "https://github.com/RWTH-EBC/AixLib.git"

# Clone into a temp folder
tmp_dir = tempfile.mkdtemp()
Repo.clone_from(REPO_URL, tmp_dir)
print(f"Cloned into {tmp_dir}")

Cloned into /var/folders/8w/nhzjf7wn3f7bxb6vlmbqbwfc0000gn/T/tmpbvq8cze8


In [10]:
import glob
import os

# 読み込み対象の拡張子
EXTENSIONS = [".md", ".py", ".txt", ".rst"]

def load_repo_texts(root_dir):
    docs = []
    for ext in EXTENSIONS:
        pattern = os.path.join(root_dir, "**", f"*{ext}")
        for path in glob.glob(pattern, recursive=True):
            # ファイルでなければスキップ
            if not os.path.isfile(path):
                continue
            # サイズが大きすぎるものはスキップ
            if os.path.getsize(path) > 1e6:
                continue
            try:
                with open(path, encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                docs.append({
                    "path": os.path.relpath(path, root_dir),
                    "content": text
                })
            except Exception as e:
                # 万が一の読み込みエラーも無視
                print(f"Warning: failed to read {path}: {e}")
    return docs

# 使い方
documents = load_repo_texts(tmp_dir)
print(f"Loaded {len(documents)} files")

Loaded 1023 files


In [11]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = []
for doc in documents:
    texts = splitter.split_text(doc["content"])
    for i, txt in enumerate(texts):
        chunks.append({
            "id": f"{doc['path']}-{i}",
            "text": txt
        })

print(f"Created {len(chunks)} text chunks")

Created 23751 text chunks


In [ ]:
import openai
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

openai.api_key = os.getenv("XAI_API_KEY")
embedder = OpenAIEmbeddings()

# texts: list of strings
texts = [c["text"] for c in chunks]
metadatas = [{"source": c["id"]} for c in chunks]

# Create FAISS index
index = FAISS.from_texts(texts, embedder, metadatas=metadatas)

In [6]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# ↓ すでに保存してある faiss_index フォルダを読み込む
embedder = OpenAIEmbeddings()
index = FAISS.load_local(
    "faiss_index",
    embedder,
    allow_dangerous_deserialization=True  # <-- これを追加
)

/var/folders/8w/nhzjf7wn3f7bxb6vlmbqbwfc0000gn/T/ipykernel_93848/3408129059.py:5: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedder = OpenAIEmbeddings()


RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char *) at /Users/runner/work/faiss-wheels/faiss-wheels/faiss/faiss/impl/io.cpp:68: Error: 'f' failed: could not open faiss_index/index.faiss for reading: No such file or directory

In [5]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# LLM
llm = ChatOpenAI(model_name="grok-3-beta",   
                 api_key=openai.api_key ,
                 base_url="https://api.x.ai/v1",
                 temperature=0)

# RetrievalQA
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",         # or "map_reduce", "refine", etc.
    retriever=index.as_retriever(),
    return_source_documents=True
)

# Ask a question
query = "Where is heat exchanger models in this project?"
result = qa(query)

print("Answer:\n", result["result"])
print("\nSource Chunks:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

NameError: name 'openai' is not defined

In [17]:
# Ask a question
query = "Construct room air conditioner model by AixLib."
result = qa(query)

print("Answer:\n", result["result"])
print("\nSource Chunks:")
for doc in result["source_documents"]:
    print(f"- {doc.metadata['source']}")

Answer:
 To construct a room air conditioner model using **AixLib**, you can leverage the library's components for HVAC systems, particularly focusing on models related to compressors, heat exchangers, and air handling. AixLib, developed at RWTH Aachen University's E.ON Energy Research Center, provides a comprehensive set of models for building performance simulations, including HVAC systems. Below, I will outline a general approach to constructing a room air conditioner model using components from AixLib. Since the provided context does not include a specific pre-built air conditioner model, we will build one using relevant sub-components.

### Step-by-Step Guide to Construct a Room Air Conditioner Model in AixLib

1. **Understand the Components of a Room Air Conditioner**:
   A typical room air conditioner (split or window unit) operates on a vapor-compression refrigeration cycle and includes:
   - A compressor (to compress the refrigerant).
   - An evaporator (to cool the room air).

## LangGraph agent

In [4]:
from langchain.chat_models import ChatOpenAI
from langgraph.graph import StateGraph
from langgraph.graph import Node
from typing import TypedDict

# Define your application state
class AgentState(TypedDict):
    # original user instruction
    instruction: str
    # Modelica components retrieved (metadata + code)
    components: list[dict]
    # Planned assembly steps
    plan: str
    # Generated Modelica model
    model_code: str

# LLM client (reuse your existing grok-3-beta config)
llm = ChatOpenAI(
    model_name="grok-3-beta",
    api_key=openai.api_key,
    base_url="https://api.x.ai/v1",
    temperature=0
)

# Node 1: Retrieve relevant Modelica components via your RAG setup
async def retrieve_components(state: AgentState) -> AgentState:
    # use your existing RetrievalQA chain
    query = f"List the component definitions relevant to: {state['instruction']}"
    result = await qa.arun(query)
    # parse the result into a list of metadata dicts
    # e.g. [{'name': 'HeatExchanger', 'code': '...'}, {'name': 'Compressor', 'code': '...'}]
    state['components'] = parse_components(result)
    return state

# Node 2: Plan how to assemble the new model
async def plan_model(state: AgentState) -> AgentState:
    prompt = (
        "Given these components:\n"
        "```\n"
        "{components}\n"
        "```\n"
        "And the user wants: {instruction}\n"
        "Generate a step-by-step plan to assemble the Modelica model."
    ).format(
        components=state['components'],
        instruction=state['instruction']
    )
    plan = await llm.apredict(messages=[{"role": "user", "content": prompt}])
    state['plan'] = plan
    return state

# Node 3: Generate the new Modelica model
async def generate_model(state: AgentState) -> AgentState:
    prompt = (
        "Using the plan below and the components provided, produce the full Modelica model code:\n"
        "Plan:\n{plan}\nComponents:\n{components}\n"        
    ).format(
        plan=state['plan'],
        components=state['components']
    )
    model_code = await llm.apredict(messages=[{"role": "user", "content": prompt}])
    state['model_code'] = model_code
    return state

# Build the graph
graph = StateGraph(state_type=AgentState)

# Register nodes
graph.add_node(Node(name="retrieve", fn=retrieve_components))
graph.add_node(Node(name="plan", fn=plan_model))
graph.add_node(Node(name="generate", fn=generate_model))

# Define edges (workflow)
graph.add_edge(source="retrieve", destination="plan")
graph.add_edge(source="plan", destination="generate")

# Optionally, add start/end markers
graph.add_preset_start("retrieve")
graph.add_preset_end("generate")

# Invoke the graph with user instruction
initial_state: AgentState = {"instruction": "Construct room air conditioner model.",
                               "components": [],
                               "plan": "",
                               "model_code": ""}

result_state = graph.invoke(initial_state)
print(result_state['model_code'])


ModuleNotFoundError: No module named 'langgraph'